# WSCC 9bus Ideal Voltage Source - EMT vs. DP

In [1]:
import requests
import glob

def download_grid_data(name, url):
    with open(name, 'wb') as out_file:
        content = requests.get(url, stream=True).content
        out_file.write(content)

url = 'https://raw.githubusercontent.com/dpsim-simulator/cim-grid-data/master/WSCC-09/WSCC-09/WSCC-09'
filename = 'WSCC-09'
download_grid_data(filename+'_EQ.xml', url+'_EQ.xml')
download_grid_data(filename+'_TP.xml', url+'_TP.xml')
download_grid_data(filename+'_SV.xml', url+'_SV.xml')

files = glob.glob(filename+'_*.xml')
print(files)

['WSCC-09_EQ.xml', 'WSCC-09_TP.xml', 'WSCC-09_SV.xml']


In [2]:
from villas.dataprocessing.readtools import *
from villas.dataprocessing.timeseries import *
import matplotlib.pyplot as plt
import re
import numpy as np
import math
import os
import subprocess

# %matplotlib widget

PEAK1PH_TO_RMS3PH=np.sqrt(3./2.)

name = 'DP_WSCC-9bus_IdealVS'
name_emt = 'EMT_WSCC-9bus_IdealVS'

timestep = 10e-6
duration = 0.1

root_path = subprocess.Popen(['git', 'rev-parse', '--show-toplevel'], stdout=subprocess.PIPE).communicate()[0].rstrip().decode('utf-8')
path_exec = root_path + '/build/dpsim/examples/cxx/'

## Run Simulation

In [3]:
sim = subprocess.Popen([path_exec+name, '--timestep', str(timestep), '--duration', str(duration), files[0], files[1], files[2]], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(sim.communicate()[0].decode())
sim = subprocess.Popen([path_exec+name_emt, '--timestep', str(timestep), '--duration', str(duration), files[0], files[1], files[2]], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(sim.communicate()[0].decode())

CIMContentHandler: Note: 0 out of 349 tasks remain unresolved!
[18:09:55.341817 DP_WSCC-9bus_IdealVS_PF info] Initialize simulation: DP_WSCC-9bus_IdealVS_PF
[18:09:55.365301 DP_WSCC-9bus_IdealVS_PF info] Scheduling tasks.
[18:09:55.365775 DP_WSCC-9bus_IdealVS_PF info] Scheduling done.
[18:09:55.365782 DP_WSCC-9bus_IdealVS_PF info] Opening interfaces.
[18:09:55.365783 DP_WSCC-9bus_IdealVS_PF info] Start synchronization with remotes on interfaces
[18:09:55.365783 DP_WSCC-9bus_IdealVS_PF info] Synchronized simulation start with remotes
[18:09:55.365784 DP_WSCC-9bus_IdealVS_PF info] Start simulation: DP_WSCC-9bus_IdealVS_PF
[18:09:55.365794 DP_WSCC-9bus_IdealVS_PF info] Time step: 1.000000e-01
[18:09:55.365798 DP_WSCC-9bus_IdealVS_PF info] Final time: 2.000000e-01
DP_WSCC-9bus_IdealVS: /usr/include/eigen3/Eigen/src/Core/Product.h:96: Eigen::Product<Lhs, Rhs, Option>::Product(const Lhs&, const Rhs&) [with _Lhs = Eigen::Matrix<std::complex<double>, -1, -1>; _Rhs = Eigen::Matrix<std::complex<

## Read DPsim Results

In [4]:
model_name = 'DP_WSCC-9bus_IdealVS'
path = 'logs/' + model_name + '/'
dpsim_result_file = path  + model_name + '.csv'
ts_dpsim_dp = read_timeseries_csv(dpsim_result_file)

model_name = 'EMT_WSCC-9bus_IdealVS'
path = 'logs/' + model_name + '/'
dpsim_result_file = path  + model_name + '.csv'
ts_dpsim_emt = read_timeseries_csv(dpsim_result_file)

FileNotFoundError: [Errno 2] No such file or directory: 'logs/DP_WSCC-9bus_IdealVS/DP_WSCC-9bus_IdealVS.csv'

## Bus voltages

In [ ]:
plt.figure(figsize=(12,8))
for name in ['v1', 'v2', 'v3']:
    plt.plot(ts_dpsim_dp[name].time, ts_dpsim_dp[name].abs().values, label=name + '(DP)')
for phase in ['0']:
    for name in ['v1_'+phase, 'v2_'+phase, 'v3_'+phase]:
        plt.plot(ts_dpsim_emt[name].time, PEAK1PH_TO_RMS3PH*ts_dpsim_emt[name].values, label=name + '(EMT)')
plt.legend()
plt.show()

## Assert bus voltages

In [ ]:
ts_emt_rms3ph = {}
rmse_rel = {}
for name in ['v1', 'v2', 'v3']:
    ts_emt_rms3ph[name+'_0'] = ts_dpsim_emt[name+'_0']
    ts_emt_rms3ph[name+'_0'].values = PEAK1PH_TO_RMS3PH*ts_emt_rms3ph[name+'_0'].values
    rmse_rel[name] = ts_dpsim_dp[name].rmse(ts_emt_rms3ph[name+'_0'], ts_dpsim_dp[name].interpolate(timestep).frequency_shift(60))/np.max(ts_dpsim_dp[name].abs().values)
    print('Rel. RMSE for {}: {}'.format(name, rmse_rel[name]))
    assert(rmse_rel[name]<1e-5)

## Bus angles from DP

In [ ]:
plt.figure(figsize=(12,8))
for name in ['v1', 'v2', 'v3']:
    plt.plot(ts_dpsim_dp[name].time, ts_dpsim_dp[name].phase().values, label=name + '(DP)')

## Generator current

In [ ]:
plt.figure(figsize=(12,8))
for phase in ['0']:
    for name in ['GEN1.I_'+phase, 'GEN2.I_'+phase, 'GEN3.I_'+phase]:
        plt.plot(ts_dpsim_emt[name].time, PEAK1PH_TO_RMS3PH*ts_dpsim_emt[name].values, label=name + '(EMT)')
for name in ['GEN1.I', 'GEN2.I', 'GEN3.I']:
    plt.plot(ts_dpsim_dp[name].time, ts_dpsim_dp[name].frequency_shift(60).values, label=name + '(DP)', linestyle='--')
plt.legend()
plt.show()

## Assert generator current

In [ ]:
ts_emt_rms3ph = {}
rmse_rel = {}
for name in ['GEN1.I', 'GEN2.I', 'GEN3.I']:
    ts_emt_rms3ph[name+'_0'] = ts_dpsim_emt[name+'_0']
    ts_emt_rms3ph[name+'_0'].values = PEAK1PH_TO_RMS3PH*ts_emt_rms3ph[name+'_0'].values
    rmse_rel[name] = ts_dpsim_dp[name].rmse(ts_emt_rms3ph[name+'_0'], ts_dpsim_dp[name].interpolate(timestep).frequency_shift(60))/np.max(ts_dpsim_dp[name].abs().values)
    print('Rel. RMSE for {}: {}'.format(name, rmse_rel[name]))
    assert(rmse_rel[name]<1e-2)